In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
%matplotlib inline
import matplotlib.pyplot as plt
import nengo
import nengo_dl

## reproducibility

In [ ]:
seed = 0
np.random.seed(seed)
torch.manual_seed(seed)

## 1.  Load MNIST

In [ ]:
print("Loading MNIST …")
to_tensor = transforms.ToTensor()
train_ds = torchvision.datasets.MNIST("/tmp/mnist_data", train=True,
                                      download=True, transform=to_tensor)
test_ds  = torchvision.datasets.MNIST("/tmp/mnist_data", train=False,
                                      download=True, transform=to_tensor)

train_images = train_ds.data.numpy().astype(np.float32) / 255.0
test_images  = test_ds.data.numpy().astype(np.float32)  / 255.0
train_labels = train_ds.targets.numpy()
test_labels  = test_ds.targets.numpy()

# Flatten 28×28 → 784
train_images_flat = train_images.reshape(len(train_images), 784)
test_images_flat  = test_images.reshape(len(test_images), 784)

print(f"Train: {train_images.shape}   Test: {test_images.shape}")

## 2.  Build & train a PyTorch MLP

In [ ]:
# The model uses only nn.Linear + nn.ReLU + nn.Flatten — all layers that
# nengo_dl.Converter can convert natively, with no fallback required.

class MnistMLP(nn.Module):
    """Flatten → Dense(512, ReLU) → Dense(10)"""
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(784, 512)
        self.act1    = nn.ReLU()
        self.fc2     = nn.Linear(512, 10)

    def forward(self, x):
        return self.fc2(self.act1(self.fc1(self.flatten(x))))

model = MnistMLP()
print(f"\nModel:\n{model}")

loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()
for epoch in range(2):
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        opt.zero_grad()
        logits = model(imgs)
        loss   = F.cross_entropy(logits, lbls)
        loss.backward(); opt.step()
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += len(imgs)
    print(f"Epoch {epoch+1}: loss={total_loss/n:.4f}  acc={correct/n*100:.1f}%")

model.eval()
torch.save(model.state_dict(), "/tmp/mnist_mlp.pt")

# PyTorch test accuracy
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=256)
correct = 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        correct += (model(imgs).argmax(1) == lbls).sum().item()
pt_acc = correct / len(test_ds)
print(f"PyTorch test accuracy: {pt_acc * 100:.2f}%")

## 3.  Convert to Nengo rate network

In [ ]:
print("\n── Converting to Nengo rate network ──")
# scale_firing_rates controls the maximum firing rate used by the spiking
# version of the converted ReLU layers.  The neuron amplitude rescales outputs
# so rate-mode probes still match the original PyTorch activations.
scale = 500
n_test = 400


def softmax_np(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def run_network(
    activation_type="rectified_linear",
    n_steps=30,
    scale_firing_rates=scale,
    synapse=None,
    n_test=n_test,
    params_file=None,
    plot=True,
):
    """Run a converted PyTorch model and plot diagnostics like keras-to-snn."""

    converter = nengo_dl.Converter(
        model,
        scale_firing_rates=scale_firing_rates,
        activation_type=activation_type,
        synapse=synapse,
    )
    nengo_input = list(converter.inputs.values())[0]
    nengo_output = list(converter.outputs.values())[-1]

    # The Keras example probes conv0.  This PyTorch model is an MLP, so the
    # equivalent early activity signal is the first dense layer output (fc1).
    hidden = converter.outputs[model.fc1]
    sample_neurons = np.linspace(
        0,
        hidden.size_in,
        min(1000, hidden.size_in),
        endpoint=False,
        dtype=np.int32,
    )

    with converter.net:
        activity_probe = nengo.Probe(hidden[sample_neurons], synapse=None)
        output_probe = nengo.Probe(nengo_output, synapse=None)
        nengo_dl.configure_settings(stateful=False)

    tiled_test_images = np.tile(
        test_images_flat[:n_test].reshape(n_test, 1, 784), (1, n_steps, 1)
    )
    inference_mode = "spiking" if activation_type == "spiking_relu" else "rate"

    minibatch_size = min(10, n_test)
    with nengo_dl.Simulator(
        converter.net, minibatch_size=minibatch_size, seed=seed, progress_bar=False
    ) as sim:
        if params_file is not None:
            sim.load_params(params_file)
        sim.run_steps(
            n_steps,
            data={nengo_input: tiled_test_images},
            inference_mode=inference_mode,
        )
        outputs = sim.data[output_probe]
        activities = sim.data[activity_probe]
        dt = sim.dt

    predictions = np.argmax(outputs[:, -1], axis=-1)
    accuracy = (predictions == test_labels[:n_test]).mean()
    print(f"Test accuracy: {100 * accuracy:.2f}%")

    if plot:
        is_spiking = activation_type == "spiking_relu"
        for ii in range(min(3, n_test)):
            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.title("Input image")
            plt.imshow(test_images[ii].reshape((28, 28)), cmap="gray")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            scaled_data = activities[ii] * scale_firing_rates
            if is_spiking:
                scaled_data = scaled_data * dt
                rates = np.sum(scaled_data, axis=0) / (n_steps * dt)
                plt.ylabel("Number of spikes")
            else:
                rates = scaled_data
                plt.ylabel("Firing rates (Hz)")
            plt.xlabel("Timestep")
            plt.title(
                f"Neural activities (fc1 mean={rates.mean():.1f} Hz, "
                f"max={rates.max():.1f} Hz)"
            )
            plt.plot(scaled_data)

            plt.subplot(1, 3, 3)
            plt.title("Output predictions")
            plt.plot(softmax_np(outputs[ii]))
            plt.legend([str(j) for j in range(10)], loc="upper left")
            plt.xlabel("Timestep")
            plt.ylabel("Probability")

            plt.tight_layout()

    return accuracy, {"outputs": outputs, "activities": activities}


rate_converter = nengo_dl.Converter(model, scale_firing_rates=scale)
rate_inp = list(rate_converter.inputs.values())[0]
rate_out = list(rate_converter.outputs.values())[-1]

with rate_converter.net:
    p_rate = nengo.Probe(rate_out, synapse=None)

print(f"Total Nengo objects: {len(list(rate_converter.net.all_objects))}")

## 4.  Run converted network in rate mode — should match PyTorch

In [ ]:
n_steps = 1
x_test_nd = test_images_flat[:n_test].reshape(n_test, 1, 784)

with nengo_dl.Simulator(rate_converter.net, minibatch_size=100, seed=seed) as sim:
    sim.run_steps(n_steps, data={rate_inp: x_test_nd}, inference_mode="rate")
    rate_preds = sim.data[p_rate][:, -1, :]

rate_acc = (np.argmax(rate_preds, axis=-1) == test_labels[:n_test]).mean()
print(f"Rate-mode accuracy:          {rate_acc * 100:.2f}%  (PyTorch: {pt_acc * 100:.2f}%)")
print()
print("Reference-style diagnostic plots for the rate network:")
rate_plot_acc, rate_plot_data = run_network(
    activation_type="rectified_linear",
    n_steps=10,
    scale_firing_rates=scale,
)

## 5.  Switch to spiking neurons — accuracy drops without filtering

In [ ]:
print("\n── Spiking mode (no synapse) ──")
# SpikingRectifiedLinear neurons: rate mode = exact ReLU; spiking mode = spikes.
# With no synaptic filtering, accuracy is measured from the final timestep.
filter_accs = {}
spike_acc, spike_plot_data = run_network(
    activation_type="spiking_relu",
    n_steps=30,
    scale_firing_rates=scale,
    synapse=None,
)
filter_accs[None] = spike_acc

for n_pres in [1, 10, 50]:
    acc, _ = run_network(
        activation_type="spiking_relu",
        n_steps=n_pres,
        scale_firing_rates=scale,
        synapse=None,
        plot=False,
    )
    print(f"  n_steps={n_pres:3d}: spiking accuracy = {acc * 100:.2f}%")

## 6.  Effect of synaptic filtering — accuracy recovers

In [ ]:
print("\n── Synaptic filtering ──")
# Like the Keras example, show the three-panel plots for each synaptic filter.
n_pres_filt = 60
for tau in [0.001, 0.005, 0.01]:
    print(f"Synapse={tau:.3f}")
    acc, _ = run_network(
        activation_type="spiking_relu",
        n_steps=n_pres_filt,
        scale_firing_rates=scale,
        synapse=tau,
    )
    filter_accs[tau] = acc
    plt.show()

## 7.  Firing rates — post-training scaling

In [ ]:
print("\n── Post-training firing-rate scaling ──")
# Higher firing rates make the spiking model more closely approximate the
# non-spiking ReLU model, at the cost of denser spike traffic.
scaling_accs = {}
for scale_i in [100, 250, 500]:
    print(f"Scale={scale_i}")
    acc, _ = run_network(
        activation_type="spiking_relu",
        n_steps=30,
        scale_firing_rates=scale_i,
        synapse=0.01,
    )
    scaling_accs[scale_i] = acc
    plt.show()

print("Scale=1000")
scale1000_acc, _ = run_network(
    activation_type="spiking_relu",
    n_steps=10,
    scale_firing_rates=1000,
)
scaling_accs[1000] = scale1000_acc

## 8.  Fine-tune with firing-rate regularisation

In [ ]:
print("\n── Fine-tuning with firing-rate regularisation ──")
target_rate = 250.0   # desired mean firing rate (Hz)
# In nengo-dl, probe values equal amplitude * rate = rate/scale (normalised units).
# The TargetFiringRate MSE loss operates in these probe units, so convert Hz → units:
target_rate_norm = target_rate / scale   # = 250/500 = 0.5

# Build a converter identical to the best spiking baseline (synapse=0.005).
ft_converter = nengo_dl.Converter(
    model, scale_firing_rates=scale,
    activation_type="spiking_relu", synapse=0.005
)
ft_inp = list(ft_converter.inputs.values())[0]
ft_out = list(ft_converter.outputs.values())[-1]

all_outs = list(ft_converter.outputs.values())
with ft_converter.net:
    p_ft_final = nengo.Probe(ft_out, synapse=None)    # for training loss
    p_ft_eval  = nengo.Probe(ft_out, synapse=0.005)   # for spiking evaluation
    p_ft_mid   = nengo.Probe(all_outs[0], synapse=None)

# Use the full training set so fine-tuning doesn't underfit.
n_train = len(train_images_flat)
x_train = train_images_flat.reshape(n_train, 1, 784)
y_train = np.eye(10, dtype=np.float32)[train_labels].reshape(n_train, 1, 10)

n_mid_neurons = p_ft_mid.size_in
y_rate = np.full((n_train, 1, n_mid_neurons), target_rate_norm, dtype=np.float32)

loss_dict   = {p_ft_final: nengo_dl.losses.CrossEntropy(),
               p_ft_mid:   nengo_dl.losses.TargetFiringRate()}
weight_dict = {p_ft_final: 1.0, p_ft_mid: 1e-3}

with nengo_dl.Simulator(ft_converter.net, minibatch_size=100, seed=seed) as sim:
    sim.compile(optimizer="adam", loss=loss_dict, loss_weights=weight_dict)
    history = sim.fit(
        x={ft_inp: x_train},
        y={p_ft_final: y_train, p_ft_mid: y_rate},
        n_steps=1,
        epochs=3,
    )
    sim.save_params("/tmp/pytorch_snn_finetuned")
    print("Fine-tuned params saved to /tmp/pytorch_snn_finetuned.npz")

    # Rate-mode accuracy after fine-tuning
    sim.run_steps(1, data={ft_inp: x_test_nd}, inference_mode="rate")
    ft_rate_preds = sim.data[p_ft_final][:, -1, :]

ft_rate_acc = (np.argmax(ft_rate_preds, axis=-1) == test_labels[:n_test]).mean()
print(f"Fine-tuned rate-mode accuracy: {ft_rate_acc * 100:.2f}%")

# Note: spiking accuracy after fine-tuning requires training *with* spiking
# dynamics (n_steps > 1) to fully recover; rate-mode fine-tuning alone adjusts
# firing rates and helps classification when longer presentation is used.

## 9.  Examine the fine-tuned model

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["loss"])
plt.title("Fine-tuning loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

print("Fine-tuned rate network")
ft_rate_plot_acc, _ = run_network(
    activation_type="rectified_linear",
    params_file="/tmp/pytorch_snn_finetuned",
    n_steps=10,
    scale_firing_rates=scale,
    synapse=0.005,
)
plt.show()

print("Fine-tuned spiking network")
ft_spike_acc, _ = run_network(
    activation_type="spiking_relu",
    params_file="/tmp/pytorch_snn_finetuned",
    n_steps=30,
    scale_firing_rates=scale,
    synapse=0.005,
)
plt.show()

## Summary

In [ ]:
print(f"""
Results ({n_test} test samples)
──────────────────────────────
PyTorch MLP accuracy:                   {pt_acc * 100:.2f}%
Converted rate-mode accuracy:           {rate_acc * 100:.2f}%
Spiking, no filter (30 steps):           {filter_accs[None] * 100:.2f}%
Spiking, τ=0.005  ({n_pres_filt} steps):          {filter_accs[0.005] * 100:.2f}%
Spiking, τ=0.01   ({n_pres_filt} steps):          {filter_accs[0.01] * 100:.2f}%
Fine-tuned rate-mode accuracy:          {ft_rate_acc * 100:.2f}%
Fine-tuned spiking accuracy:            {ft_spike_acc * 100:.2f}%

Key takeaways
─────────────
• nengo_dl.Converter maps nn.Linear + nn.ReLU → nengo.Ensemble with
  RectifiedLinear (rate) or SpikingRectifiedLinear (spiking) neurons.
• The diagnostic plots mirror the Keras-to-SNN example: input image,
  early-layer neural activity, and output probabilities over time.
• Spiking neurons trade accuracy for sparse spike-based communication;
  synaptic filtering and higher firing rates help recover the rate model.
• Fine-tuning with nengo-dl's sim.fit() adjusts weights to work better at the
  target firing rate, and the same plotting helper can inspect the result.
""")